# EvalPlus benchmark: Qwen2.5-Coder-7B v4 (merged) — HumanEval/HumanEval+ và MBPP/MBPP+, prompt base và ChatML

Gộp `run-evalplus-kaggle.ipynb` (HumanEval) và `run-evalplus-mbpp.ipynb` (MBPP) thành một notebook. vLLM, greedy pass@1.
Mỗi model chạy 2 dataset × 2 kiểu prompt = 4 lần; mỗi lần EvalPlus báo cả bản gốc và bản `+` (HE, HE+, MBPP, MBPP+).

| Kiểu prompt | Cờ EvalPlus | Prompt gửi cho model |
|---|---|---|
| `base` | `--force-base-prompt` | completion thuần: signature + docstring, model viết tiếp |
| `chatml` | (không cờ) | chat template của tokenizer (ChatML, kèm system prompt mặc định của Qwen) |

**Trước khi chạy (Kaggle):**
1. Settings → Accelerator: **GPU T4 x2**. Settings → Internet: **On**.
2. Add-ons → Secrets: thêm `HF_TOKEN` (quyền đọc, repo `-merged` đang private) và attach vào notebook.

Model 7B fp16 (~15GB) không vừa 1 T4 nên dùng `--tp 2`; T4 không có bf16 nên dùng `float16`.

**Lưu ý quan trọng:** EvalPlus đặt tên file kết quả chỉ theo tên model, không theo kiểu prompt, và mặc định *resume* (dùng lại mẫu và điểm đã có). Chạy cùng model với hai kiểu prompt vào cùng thư mục thì lần sau sẽ lặp lại kết quả của lần trước. Notebook này tách thư mục kết quả theo kiểu prompt (`--root evalplus_results/<prompt>`) để tránh lỗi đó.

## 1. Cài đặt

In [1]:
!pip install -q uv
!uv venv /kaggle/working/vllm_env --python 3.12
!uv pip install -q --python /kaggle/working/vllm_env/bin/python "evalplus[vllm]"
!/kaggle/working/vllm_env/bin/python -c "import torch, vllm; print('torch', torch.__version__, 'cuda', torch.version.cuda, '| vllm', vllm.__version__)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 82.1 MB/s eta 0:00:00
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: vllm_env
Activate with: source vllm_env/bin/activate
torch 2.13.0+cu130 cuda 13.0 | vllm 0.32.0


## 2. Config

In [2]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

MODELS = {
    "v4": "hoangnh39/qwen2.5-coder-7b-opencode-v4-merged",
}
DATASETS = ["humaneval", "mbpp"]   # mỗi dataset báo cả bản gốc và bản + (HE/HE+, MBPP/MBPP+)
PROMPTS = {                        # tên -> cờ EvalPlus thêm vào
    "base": "--force-base-prompt",
    "chatml": "",
}
TP, DTYPE = 2, "float16"
EVAL = "/kaggle/working/vllm_env/bin/evalplus.evaluate"

!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


## 3. Kiểm tra trước khi chạy

EvalPlus chỉ dùng chat template khi tokenizer của model có `chat_template`; thiếu thì kiểu `chatml` âm thầm chạy giống `base`. Cell này dừng sớm nếu thiếu, và in token dừng của model.

In [3]:
from transformers import AutoTokenizer, GenerationConfig

for tag, model in MODELS.items():
    tok = AutoTokenizer.from_pretrained(model)
    assert tok.chat_template, f"{model}: tokenizer thiếu chat_template -> kiểu chatml sẽ chạy như base"
    print(f"{tag}: chat_template OK | generation_config eos = {GenerationConfig.from_pretrained(model).eos_token_id}")
    print(tok.apply_chat_template([{"role": "user", "content": "hi"}], tokenize=False, add_generation_prompt=True))

config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/722 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/2.51k [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/139 [00:00<?, ?B/s]

v4: chat_template OK | generation_config eos = [151645, 151643]
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
hi<|im_end|>
<|im_start|>assistant



## 4. Chạy benchmark

Mỗi lần chạy là một tiến trình riêng nên GPU được giải phóng giữa các lần. Log: `evalplus_<tag>_<prompt>_<dataset>.log`; mẫu và kết quả chi tiết: `evalplus_results/<prompt>/<dataset>/`.

In [4]:
for tag, model in MODELS.items():
    for ds in DATASETS:
        for prompt, flag in PROMPTS.items():
            print(f"===== {tag} / {ds} / {prompt} =====", flush=True)
            !{EVAL} --model {model} --dataset {ds} --backend vllm --greedy {flag} --root evalplus_results/{prompt} --tp {TP} --dtype {DTYPE} 2>&1 | tee evalplus_{tag}_{prompt}_{ds}.log | tail -n 12

===== v4 / humaneval / base =====
(EngineCore pid=221) INFO 10-28 03:34:06 [multiproc_executor.py:479] [shutdown] Executor: waiting for worker exit count=2
(Worker_TP0 pid=229) INFO 10-28 03:34:06 [multiproc_executor.py:843] Parent process exited, terminating worker queues
(EngineCore pid=221) INFO 10-28 03:34:08 [multiproc_executor.py:486] [shutdown] Executor: all workers exited gracefully
Computing expected output...
Expected outputs computed in 24.85s
Reading samples...
164it [00:00, 1973.34it/s]
100%|██████████| 164/164 [00:20<00:00,  8.03it/s]
humaneval (base tests)
pass@1:	0.750
humaneval+ (base + extra tests)
pass@1:	0.683
===== v4 / humaneval / chatml =====
(MainProcess pid=1002) WARNING 10-28 04:06:22 [system_utils.py:225] [shutdown] force killing sub process VLLM::Worker_TP pid 1038
(MainProcess pid=1002) WARNING 10-28 04:06:22 [system_utils.py:225] [shutdown] force killing sub process VLLM::Worker_TP1 pid 1039
(EngineCore pid=1030) WARNING 10-28 04:06:22 [multiproc_executor.

## 5. Summary

In [5]:
import re, glob
import pandas as pd

ANSI = re.compile(r"\x1b\[[0-9;]*m")
SCORE = re.compile(r"(humaneval|mbpp)(\+?) \((?:base tests|base \+ extra tests)\)\s*pass@1:\s*([\d.]+)")
COLS = {"humaneval": "HE", "humaneval+": "HE+", "mbpp": "MBPP", "mbpp+": "MBPP+"}

rows = {}
for path in sorted(glob.glob("evalplus_*_*_*.log")):
    tag, prompt, _ = re.fullmatch(r"evalplus_(.+?)_(base|chatml)_(humaneval|mbpp)\.log", path).groups()
    for name, plus, score in SCORE.findall(ANSI.sub("", open(path).read())):
        rows.setdefault((tag, prompt), {})[COLS[name + plus]] = round(float(score) * 100, 1)

table = pd.DataFrame(rows).T.reindex(columns=list(COLS.values()))
table.index.names = ["model", "prompt"]
table["avg"] = table.mean(axis=1).round(1)
print(table.to_string())
table.to_csv("evalplus_summary.csv")
try:
    print("\n" + table.to_markdown())   # cần tabulate
except ImportError:
    pass

                HE   HE+  MBPP  MBPP+   avg
model prompt                               
v4    base    75.0  68.3  84.7   70.1  74.5
      chatml  86.0  79.3  79.6   68.3  78.3

|                  |   HE |   HE+ |   MBPP |   MBPP+ |   avg |
|:-----------------|-----:|------:|-------:|--------:|------:|
| ('v4', 'base')   |   75 |  68.3 |   84.7 |    70.1 |  74.5 |
| ('v4', 'chatml') |   86 |  79.3 |   79.6 |    68.3 |  78.3 |
